# nb12.1 — Submission Packet Builder: One Zip, One Manifest, One SHA-256

*Week 12, Chapter 12.1.* You wrote the paper, cleaned the code, drafted the abstract.
Now the Schar Symposium portal asks for **a single zip** with an exact set of files
and the question on the form reads, in tiny grey print, *"Does your submission match
your submitted manifest?"* The answer needs to be yes, and you need to be able to
prove it three weeks later when a faculty reviewer emails asking which version of
Table 3 corresponds to which line in the regression script.

That is what this notebook builds: a `pandas`-driven manifest that assembles your
paper PDF, the replication packet, your CV, an AI-use disclosure, and your ORCID
into one Schar-ready bundle whose contents are cryptographically pinned by a
SHA-256 hash. The trick is small — a manifest is just a CSV with a hash column —
but the discipline matters: once a manifest exists, nobody (including you, six
months from now) can quietly swap a figure without leaving a fingerprint.


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless render, never to a screen

import hashlib
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # seed once, everywhere

# Work in a sandbox directory so the notebook never touches your real submissions.
WORK = Path("/tmp/nb12_1_packet")
WORK.mkdir(exist_ok=True)
print("workspace:", WORK)


## 1. The five required artifacts

A Schar-ready bundle is not a vibe. It is exactly five things, named exactly the
way the symposium portal expects them. Maya's bundle, for the fair-lending paper
she has been building since Week 1, looks like the table below. Each row in the
manifest is one artifact; each artifact has a role, an expected filename, and a
cryptographic fingerprint.

Note the convention: `kebab-case-with-author-last-name.ext`. Schar's submission
script rejects spaces and capital letters in filenames because they break on
Windows zip extraction. We follow their rules.


In [ ]:
# Define the five required artifacts. Each row carries: role, filename, mime
# type, whether it is required, and a short note for the reviewer.
ARTIFACTS = pd.DataFrame([
    ("paper",        "rivera-fair-lending-2026.pdf",  "application/pdf", True,
     "Main manuscript, 25 pages including references and tables."),
    ("replication",  "rivera-replication-packet.zip", "application/zip", True,
     "All code, synthetic data, README, environment.yml. See nb12.3 for MARS schema."),
    ("cv",           "rivera-cv.pdf",                  "application/pdf", True,
     "One-page student CV with coursework, projects, and contact info."),
    ("ai_disclosure","rivera-ai-use-disclosure.md",   "text/markdown",   True,
     "Per Schar policy: which AI tools, for what tasks, with what verification."),
    ("orcid",        "rivera-orcid.txt",               "text/plain",      True,
     "Plain text ORCID iD: 0000-0000-0000-0000 format."),
], columns=["role", "filename", "mime", "required", "note"])

ARTIFACTS


## 2. Synthesizing the artifacts (because this is a teaching notebook)

In a real submission you would already have these files. Here we write small
plausible stand-ins so the rest of the notebook has something concrete to hash
and zip. The PDFs are not real PDFs — they are text files renamed `.pdf` — but
the hashing, manifesting, and zipping logic is identical. When you run this for
your own paper, replace these `write_text` calls with `shutil.copy(real_path, WORK / row.filename)`.


In [ ]:
STUB_CONTENT = {
    "rivera-fair-lending-2026.pdf":
        "Rivera (2026). Race, ZIP Codes, and Mortgage Denials in HMDA 2018-2023. "
        "Working paper, Schar Symposium submission. 25 pp.",
    "rivera-cv.pdf":
        "Maya Rivera. Senior, Lincoln HS. AP Calc BC, AP Stats. ASSIP 2026.",
    "rivera-ai-use-disclosure.md":
        "# AI Use Disclosure\n\nClaude Sonnet for prose polish on Sections 1 and 5; "
        "all code, regressions, and tables produced by author. Every AI-suggested edit "
        "was manually verified against the underlying data.",
    "rivera-orcid.txt":
        "0009-0001-2345-6789",
}

for fname, content in STUB_CONTENT.items():
    (WORK / fname).write_text(content, encoding="utf-8")

# Build the inner replication zip from a tiny synthetic packet
inner = WORK / "rivera-replication-packet.zip"
with zipfile.ZipFile(inner, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("README.md", "# Replication packet\n\nSee nb12.3 for full MARS schema.")
    z.writestr("environment.yml", "name: rivera\ndependencies:\n  - python=3.11\n  - pandas\n")
    z.writestr("code/01-clean-hmda.py", "# placeholder cleaning script\nprint('clean')\n")

sorted(p.name for p in WORK.iterdir())


## 3. SHA-256: a 64-character fingerprint that catches every silent change

A cryptographic hash is a function $h: \{0,1\}^* \to \{0,1\}^{256}$ that takes a
file of any length and returns a 256-bit number, written as 64 hex characters.
Two properties make it useful for replication bundles:

1. **Determinism.** The same bytes always produce the same hash. Run it today,
   run it in 2030 on a different machine — same output.
2. **Avalanche.** Flipping a single bit in the input scrambles every bit of the
   output with probability $\tfrac{1}{2}$. So if your reviewer's copy of Table 3
   differs from yours by one rounding digit, the hashes will not match and you
   will see it immediately.

We use Python's standard `hashlib.sha256` and stream the file in 64 KB chunks so
the same code works whether the artifact is 2 KB or 2 GB.


In [ ]:
def sha256_of(path, chunk=65536):
    """Return the 64-character hex SHA-256 of the file at `path`."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# Demonstrate avalanche: change one byte, watch the hash change completely
test_path = WORK / "_demo.txt"
test_path.write_text("Maya's paper, version 1")
h1 = sha256_of(test_path)
test_path.write_text("Maya's paper, version 2")  # flipped one digit
h2 = sha256_of(test_path)
print("v1 sha256:", h1)
print("v2 sha256:", h2)
print("characters that match:", sum(a == b for a, b in zip(h1, h2)), "of 64")
test_path.unlink()


Roughly four characters out of sixty-four match by chance — exactly what we
expect from a uniform random hex string. That is the avalanche property doing
its job. There is no "small change" to a SHA-256 hash; either you have the same
file or you have a different file.


## 4. Building the manifest

The manifest is a `pandas` DataFrame with one row per artifact. We add three
computed columns: the hash, the byte size, and the modification timestamp (UTC,
ISO-8601). The manifest is then written as both CSV (human-readable, opens in
Excel) and JSON (machine-readable, embeds into the outer zip).


In [ ]:
def build_manifest(artifacts, root):
    """Augment the artifact table with sha256, size_bytes, and mtime_utc."""
    out = artifacts.copy()
    hashes, sizes, mtimes, exists = [], [], [], []
    for fname in out["filename"]:
        p = root / fname
        if p.exists():
            hashes.append(sha256_of(p))
            sizes.append(p.stat().st_size)
            # Use ISO-8601 UTC for a portable, sortable timestamp string
            mtimes.append(pd.Timestamp(p.stat().st_mtime, unit="s", tz="UTC").isoformat())
            exists.append(True)
        else:
            hashes.append("")
            sizes.append(0)
            mtimes.append("")
            exists.append(False)
    out["sha256"] = hashes
    out["size_bytes"] = sizes
    out["mtime_utc"] = mtimes
    out["exists"] = exists
    return out

manifest = build_manifest(ARTIFACTS, WORK)
manifest[["role", "filename", "size_bytes", "sha256"]]


## 5. Validation: does every required artifact exist?

Before zipping, we check the manifest itself. A bundle that ships with a missing
required artifact is worse than no bundle — it implies completeness it does not
deliver. The check is two lines of `pandas` and they are the most important two
lines in this notebook.


In [ ]:
def validate_manifest(m):
    """Return a list of validation errors. Empty list means ready to ship."""
    errors = []
    missing = m.loc[m["required"] & ~m["exists"], "filename"].tolist()
    if missing:
        errors.append(f"Missing required artifacts: {missing}")
    empty_hashes = m.loc[m["exists"] & (m["sha256"] == ""), "filename"].tolist()
    if empty_hashes:
        errors.append(f"Artifacts present but unhashable: {empty_hashes}")
    # SHA-256 hex strings are exactly 64 lowercase hex characters
    bad = m.loc[m["exists"] & (m["sha256"].str.len() != 64), "filename"].tolist()
    if bad:
        errors.append(f"Malformed SHA-256 for: {bad}")
    return errors

errors = validate_manifest(manifest)
print("validation errors:", errors if errors else "(none - ready to ship)")


## 6. Writing the manifest in two formats

CSV is what the reviewer opens in Excel; JSON is what the symposium's automated
checker parses. We write both, and we include a top-level `manifest_version`
string so future Schar tooling can route old manifests to the right validator.


In [ ]:
def write_manifest(m, root, author, paper_title):
    """Write manifest.csv and manifest.json to `root` and return the json dict."""
    csv_path = root / "manifest.csv"
    json_path = root / "manifest.json"
    m.to_csv(csv_path, index=False)

    payload = {
        "manifest_version": "schar-2026-v1",
        "author": author,
        "paper_title": paper_title,
        "generated_utc": pd.Timestamp("2026-09-11T12:00:00", tz="UTC").isoformat(),
        "artifacts": m.drop(columns=["exists"]).to_dict(orient="records"),
    }
    json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return payload

payload = write_manifest(
    manifest, WORK,
    author="Maya Rivera",
    paper_title="Race, ZIP Codes, and Mortgage Denials in HMDA 2018-2023",
)
print(json.dumps(payload, indent=2)[:500], "...")


## 7. Zipping the bundle (with the manifest *inside*)

The outer zip contains: the five artifacts, the manifest in CSV form, and the
manifest in JSON form. The manifest sits **inside** the zip — that is what makes
the bundle self-verifying. Anyone who downloads it can re-hash the artifacts and
compare against the manifest without needing any external file.


In [ ]:
def build_bundle(root, manifest_df, bundle_name):
    """Zip all manifest-listed artifacts plus the manifest files into bundle_name."""
    bundle_path = root / bundle_name
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as z:
        for fname in manifest_df["filename"]:
            p = root / fname
            if p.exists():
                z.write(p, arcname=fname)
        z.write(root / "manifest.csv", arcname="manifest.csv")
        z.write(root / "manifest.json", arcname="manifest.json")
    return bundle_path

bundle = build_bundle(WORK, manifest, "rivera-schar-2026-bundle.zip")
print("bundle path:", bundle)
print("bundle size:", bundle.stat().st_size, "bytes")
with zipfile.ZipFile(bundle) as z:
    for info in z.infolist():
        print(f"  {info.filename:<40} {info.file_size:>8} bytes")


## 8. Verifying the bundle (the reviewer's-eye view)

Now flip perspective. You are a Schar reviewer. The bundle lands in your inbox.
You extract it, re-hash each artifact, and compare to `manifest.json`. If every
hash matches, the bundle is intact. If any hash mismatches, you write back to
the author the same day.


In [ ]:
def verify_bundle(bundle_path, extract_to):
    """Extract a bundle, re-hash every artifact, and return a comparison table."""
    extract_to.mkdir(exist_ok=True)
    with zipfile.ZipFile(bundle_path) as z:
        z.extractall(extract_to)
    declared = json.loads((extract_to / "manifest.json").read_text())
    rows = []
    for artifact in declared["artifacts"]:
        fname = artifact["filename"]
        declared_hash = artifact["sha256"]
        actual_hash = sha256_of(extract_to / fname) if (extract_to / fname).exists() else ""
        rows.append({
            "filename": fname,
            "declared": declared_hash[:12] + "...",
            "actual":   actual_hash[:12] + "...",
            "match":    declared_hash == actual_hash and declared_hash != "",
        })
    return pd.DataFrame(rows)

verify_dir = WORK / "_verify"
report = verify_bundle(bundle, verify_dir)
report


## 9. The tamper test (why this isn't theatre)

To show the verification step has teeth, we corrupt one artifact inside the
extracted directory — a single character changed in Maya's CV — and re-run the
check. The CV row should now flip from `match=True` to `match=False`.


In [ ]:
# Tamper with the CV after extraction
tampered = verify_dir / "rivera-cv.pdf"
tampered.write_text(tampered.read_text() + " (silently appended)")

# Re-hash and compare against the declared manifest
declared = json.loads((verify_dir / "manifest.json").read_text())
rows = []
for artifact in declared["artifacts"]:
    fname = artifact["filename"]
    declared_hash = artifact["sha256"]
    actual_hash = sha256_of(verify_dir / fname) if (verify_dir / fname).exists() else ""
    rows.append({
        "filename": fname,
        "match": declared_hash == actual_hash and declared_hash != "",
    })
tamper_report = pd.DataFrame(rows)
tamper_report


Exactly the CV row reads `False`, and the other four read `True`. The hash
caught a one-character change in a five-file bundle. That is what
cryptographic integrity buys you: a reviewer cannot accidentally compare against
a stale copy, and you cannot accidentally submit a stale copy.


## 10. When it fails (the gotchas)

Three failure modes are worth pre-empting because they will bite you at 11pm
the night before the deadline.

**Line-ending drift.** Windows writes `\r\n`, Linux writes `\n`. Open a plain-text
artifact in Windows Notepad, save it without touching anything else, and its
SHA-256 changes. Keep text artifacts opened in editors that preserve EOLs
(VS Code with `files.eol` set, not Notepad).

**Filesystem case-folding.** macOS HFS+ is case-insensitive but case-preserving;
Linux ext4 is case-sensitive. `Rivera-CV.pdf` and `rivera-cv.pdf` are the same
file on a Mac and different files on Hopper. Standardize to lowercase kebab-case.

**Timestamp leaking.** Zip metadata stores per-file modification times, so two
bundles built five minutes apart from byte-identical sources have different zip
hashes. We hash the **contents** of artifacts, not the zip itself, precisely to
sidestep this. The manifest's `mtime_utc` column is informational, not part of
the integrity guarantee.


## 11. Your turn

1. **Swap in a real artifact.** Replace the stub `rivera-cv.pdf` with your
   actual CV PDF. Re-run cells 4 through 7. Confirm the size and hash change.

2. **Add a sixth artifact.** Many programs require an advisor letter. Add a
   row to `ARTIFACTS` for `rivera-advisor-letter.pdf`, create a stub file,
   and confirm the manifest picks it up.

3. **Quarantine missing files.** Modify `validate_manifest` so that instead of
   returning errors as a list, it returns a `pd.DataFrame` with one row per
   issue and a severity column (`error` vs `warning`). Missing required artifacts
   are errors; oversized artifacts (>20 MB) are warnings.

4. **End-to-end smoke test.** Write a function `roundtrip(root) -> bool` that
   builds the bundle, extracts it to a fresh directory, verifies every hash,
   and returns `True` only if no errors and no mismatches. Run it.
